# Tutorial de Scikit-Learn: Regresión Lineal
Basado en el tutorial de **coursera's IBM Python for Data Science and AI y notas del Dr. Ivan Olier-Caparroso**

## Introducción
**Scikit-learn** (sklearn) es la librería más utilizada en Python para Machine Learning. En este notebook aprenderás:

- Cómo particionar datos en **Train / Validation / Test**
- Qué es y por qué importa la **partición estratificada**
- **Regresión Lineal Simple** (una sola variable predictora)
- **Regresión Lineal Múltiple** (varias variables predictoras)
- **Regresión con Variables Categóricas** (One-Hot Encoding)
- Evaluación del modelo con métricas de regresión

<hr/>
<div class="alert alert-success alertsuccess" style="margin-top: 20px">
[Consejo]: Ejecuta cada celda con <kbd>Shift</kbd> + <kbd>Enter</kbd>. Es importante ejecutarlas en orden.
</div>
<hr/>

In [ ]:
# Importar todas las librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos de scikit-learn que usaremos en este notebook
from sklearn.model_selection import train_test_split   # Partición de datos
from sklearn.linear_model import LinearRegression      # Modelo de regresión lineal
from sklearn.preprocessing import OneHotEncoder        # Codificación de variables categóricas
from sklearn.preprocessing import StandardScaler       # Estandarización de variables
from sklearn.metrics import mean_absolute_error        # Error absoluto medio (MAE)
from sklearn.metrics import mean_squared_error         # Error cuadrático medio (MSE)
from sklearn.metrics import r2_score                   # Coeficiente de determinación R²

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(42)

import sklearn
print('Versión de scikit-learn:', sklearn.__version__)

## 1. Creación del Dataset

Simulamos un dataset de **precios de viviendas** con las siguientes variables:

| Variable | Tipo | Descripción |
|---|---|---|
| `metros_cuadrados` | Numérica | Superficie de la vivienda |
| `habitaciones` | Numérica | Número de habitaciones |
| `antiguedad` | Numérica | Años desde construcción |
| `distancia_centro` | Numérica | Km al centro de la ciudad |
| `tipo_vivienda` | Categórica | Apartamento / Casa / Estudio |
| `barrio` | Categórica | Norte / Sur / Centro / Este |
| `precio` | Numérica | **Variable objetivo** (en miles de USD) |

In [ ]:
## Creemos una base de datos
np.random.seed(42)
n = 500

metros        = np.random.normal(90, 30, n).clip(30, 250)
habitaciones  = np.random.randint(1, 6, n)
antiguedad    = np.random.randint(0, 40, n)
distancia     = np.random.uniform(0.5, 20, n)
tipo_vivienda = np.random.choice(['Apartamento', 'Casa', 'Estudio'], n, p=[0.5, 0.35, 0.15])
barrio        = np.random.choice(['Norte', 'Sur', 'Centro', 'Este'], n, p=[0.3, 0.25, 0.3, 0.15])

# El precio depende de múltiples factores con algo de ruido
precio = (
    50                               # precio base
    + metros        * 1.8            # más metros → más caro
    + habitaciones  * 15             # más habitaciones → más caro
    - antiguedad    * 1.2            # más antiguo → más barato
    - distancia     * 4.5            # más lejos → más barato
    + (tipo_vivienda == 'Casa') * 60 # las casas valen más
    + (barrio == 'Centro') * 40      # el centro vale más
    + (barrio == 'Norte')  * 20
    + np.random.normal(0, 25, n)     # ruido aleatorio (factores no medidos)
).clip(50, 800)

df = pd.DataFrame({
    'metros_cuadrados': metros,
    'habitaciones':     habitaciones,
    'antiguedad':       antiguedad,
    'distancia_centro': distancia,
    'tipo_vivienda':    tipo_vivienda,
    'barrio':           barrio,
    'precio':           precio
})

print('Forma del dataset:', df.shape)
df.head()

In [ ]:
# Estadísticas descriptivas
df.describe().round(2)

In [ ]:
# Exploración visual rápida
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df['precio'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribución del Precio')
axes[0].set_xlabel('Precio (miles USD)')

sns.scatterplot(data=df, x='metros_cuadrados', y='precio', alpha=0.3, ax=axes[1])
axes[1].set_title('Precio vs Metros Cuadrados')

sns.boxplot(data=df, x='tipo_vivienda', y='precio', palette='Set2', ax=axes[2])
axes[2].set_title('Precio por Tipo de Vivienda')

plt.tight_layout()
plt.show()

## 2. Partición de Datos: Train / Validation / Test

### ¿Por qué dividir los datos?

Uno de los principios fundamentales del Machine Learning es evaluar el modelo con datos que **nunca ha visto** durante el entrenamiento. Si evaluamos con los mismos datos de entrenamiento, el modelo puede memorizar los datos (*overfitting*) y dar una evaluación falsamente optimista.

La división estándar es:

```
Dataset completo (100%)
├── Train      (70%) → El modelo APRENDE con estos datos
├── Validation (15%) → Ajustamos hiperparámetros y comparamos modelos
└── Test       (15%) → Evaluación FINAL, solo se usa UNA vez
```

<hr/>
<div class="alert alert-warning" style="margin-top: 20px">
⚠️ <b>Regla de oro</b>: El conjunto de Test es intocable hasta que el modelo esté completamente finalizado. Usarlo antes contamina la evaluación.
</div>
<hr/>

In [ ]:
# Separar features (X) y variable objetivo (y)
# Por ahora usamos solo variables numéricas
X_num = df[['metros_cuadrados', 'habitaciones', 'antiguedad', 'distancia_centro']]
y     = df['precio']

print('Forma de X:', X_num.shape)
print('Forma de y:', y.shape)

In [ ]:
# ── PASO 1: separar Test (15%) del resto (85%) ──────────────────────────────
# train_test_split(X, y, test_size, random_state)
#   X, y         → datos de entrada y etiquetas
#   test_size    → proporción del conjunto de prueba (0.15 = 15%)
#   random_state → semilla para reproducibilidad (cualquier entero)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_num, y,
    test_size=0.15,
    random_state=42
)

# ── PASO 2: del 85% restante, separar Validation (≈15% del total) ───────────
# 0.15 / 0.85 ≈ 0.176 para obtener ~15% del total original
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=42
)

print('Tamaños de los conjuntos:')
print(f'  Train:      {X_train.shape[0]} muestras ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'  Validation: {X_val.shape[0]}  muestras ({X_val.shape[0]/len(df)*100:.1f}%)')
print(f'  Test:       {X_test.shape[0]}  muestras ({X_test.shape[0]/len(df)*100:.1f}%)')
print(f'  Total:      {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]}')

In [ ]:
# Verificar que las distribuciones sean similares en los tres conjuntos
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, y_part, titulo, color in zip(
    axes,
    [y_train, y_val, y_test],
    ['Train', 'Validation', 'Test'],
    ['steelblue', 'orange', 'green']
):
    sns.histplot(y_part, kde=True, ax=ax, color=color, bins=20)
    ax.set_title(f'{titulo}\nMedia={y_part.mean():.1f}  Std={y_part.std():.1f}')
    ax.set_xlabel('Precio')

plt.suptitle('Distribución del Precio en cada Partición', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Regresión Lineal Simple

La **Regresión Lineal Simple** modela la relación entre una variable predictora $x$ y una variable respuesta $y$ mediante una línea recta:

$$\hat{y} = \beta_0 + \beta_1 x$$

Donde:
- $\beta_0$ = intercepto (valor de $y$ cuando $x = 0$)
- $\beta_1$ = pendiente (cuánto cambia $y$ por cada unidad de $x$)

El algoritmo busca los valores de $\beta_0$ y $\beta_1$ que **minimizan el error cuadrático** entre las predicciones y los valores reales.

In [ ]:
# ── Regresión Lineal Simple: solo 'metros_cuadrados' ────────────────────────

# Seleccionar una sola variable (sklearn requiere forma 2D: n_muestras × n_features)
X_simple_train = X_train[['metros_cuadrados']]
X_simple_val   = X_val[['metros_cuadrados']]

# 1. Instanciar el modelo
modelo_simple = LinearRegression()

# 2. Entrenar el modelo (fit) con datos de Train
#    Aquí sklearn calcula internamente β₀ y β₁
modelo_simple.fit(X_simple_train, y_train)

# 3. Ver los coeficientes aprendidos
print(f'Intercepto (β₀):           {modelo_simple.intercept_:.4f}')
print(f'Pendiente β₁ (metros²):    {modelo_simple.coef_[0]:.4f}')
print()
print('Interpretación: por cada m² adicional, el precio sube')
print(f'{modelo_simple.coef_[0]:.2f} miles de USD en promedio.')

In [ ]:
# 4. Hacer predicciones sobre Validation
y_pred_simple = modelo_simple.predict(X_simple_val)

# 5. Visualizar la recta de regresión
plt.figure(figsize=(10, 5))
plt.scatter(X_simple_val, y_val, alpha=0.4, label='Datos reales', color='steelblue')
plt.plot(
    sorted(X_simple_val['metros_cuadrados']),
    modelo_simple.predict(pd.DataFrame({'metros_cuadrados': sorted(X_simple_val['metros_cuadrados'])})),
    color='red', linewidth=2, label='Recta de regresión'
)
plt.title('Regresión Lineal Simple: Precio ~ Metros Cuadrados')
plt.xlabel('Metros Cuadrados')
plt.ylabel('Precio (miles USD)')
plt.legend()
plt.show()

## 4. Métricas de Evaluación para Regresión

Las métricas más comunes para evaluar modelos de regresión son:

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum \|y_i - \hat{y}_i\| $ | Error promedio en las mismas unidades de $y$ |
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Penaliza errores grandes más que MAE |
| **RMSE** | $\sqrt{MSE}$ | Igual que MAE pero en unidades de $y$, penaliza más los outliers |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporción de varianza explicada. Rango: $(-\infty, 1]$. Ideal: cerca de 1 |

In [ ]:
def evaluar_modelo(y_real, y_predicho, nombre_conjunto='Validation'):
    """Calcula e imprime las métricas de regresión."""
    mae  = mean_absolute_error(y_real, y_predicho)
    mse  = mean_squared_error(y_real, y_predicho)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_real, y_predicho)

    print(f'── Métricas en {nombre_conjunto} ──────────────────')
    print(f'  MAE  (Error Absoluto Medio):       {mae:.2f} miles USD')
    print(f'  MSE  (Error Cuadrático Medio):     {mse:.2f}')
    print(f'  RMSE (Raíz del MSE):               {rmse:.2f} miles USD')
    print(f'  R²   (Coef. de Determinación):     {r2:.4f}')
    print()
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

metricas_simple = evaluar_modelo(y_val, y_pred_simple)

## 5. Regresión Lineal Múltiple (solo numéricas)

Extendemos el modelo para incluir **todas las variables numéricas**:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_3 + \beta_4 x_4$$

In [ ]:
# Modelo múltiple con las 4 variables numéricas
modelo_mult = LinearRegression()
modelo_mult.fit(X_train, y_train)

# Coeficientes del modelo
coeficientes = pd.DataFrame({
    'Variable':    X_train.columns,
    'Coeficiente': modelo_mult.coef_
}).sort_values('Coeficiente', ascending=False)

print(f'Intercepto: {modelo_mult.intercept_:.4f}\n')
print(coeficientes.to_string(index=False))
print()
print('Interpretación: controlando el resto de variables,')
print('cada variable cambia el precio en su coeficiente (miles USD).')

In [ ]:
# Evaluar modelo múltiple
y_pred_mult = modelo_mult.predict(X_val)
metricas_mult = evaluar_modelo(y_val, y_pred_mult, 'Validation (múltiple)')

# Comparar con el modelo simple
print('── Comparación ─────────────────────────────────')
print(f'  R² Simple:   {metricas_simple["R2"]:.4f}')
print(f'  R² Múltiple: {metricas_mult["R2"]:.4f}  ← mejora al agregar variables')

In [ ]:
# Gráfico de valores reales vs predichos (diagnóstico clave)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, titulo in zip(axes,
                               [y_pred_simple, y_pred_mult],
                               ['Simple (1 variable)', 'Múltiple (4 variables)']):
    ax.scatter(y_val, y_pred, alpha=0.4, color='steelblue')
    lims = [min(y_val.min(), y_pred.min()), max(y_val.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
    ax.set_xlabel('Precio Real')
    ax.set_ylabel('Precio Predicho')
    ax.set_title(f'Real vs Predicho\n{titulo}')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Regresión con Variables Categóricas

Los modelos de regresión lineal **no pueden usar texto directamente**. Las variables categóricas como `tipo_vivienda` o `barrio` deben transformarse a números.

### One-Hot Encoding (OHE)

La técnica más común es **One-Hot Encoding**, que crea una columna binaria (0/1) por cada categoría:

```
tipo_vivienda          tipo_Apartamento  tipo_Casa  tipo_Estudio
──────────────    →    ────────────────  ─────────  ────────────
Apartamento                   1              0           0
Casa                          0              1           0
Estudio                       0              0           1
```

<hr/>
<div class="alert alert-warning" style="margin-top: 20px">
⚠️ <b>Dummy Variable Trap</b>: Con k categorías se crean k-1 columnas para evitar multicolinealidad perfecta. Sklearn maneja esto con el parámetro <code>drop='first'</code>.
</div>
<hr/>

In [ ]:
# ── Preparar features incluyendo variables categóricas ──────────────────────

# Primero hacemos la partición train/val/test con TODOS los features
X_full = df[['metros_cuadrados', 'habitaciones', 'antiguedad',
             'distancia_centro', 'tipo_vivienda', 'barrio']]

X_temp2, X_test2, y_temp2, y_test2 = train_test_split(
    X_full, y, test_size=0.15, random_state=42
)
X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X_temp2, y_temp2, test_size=0.176, random_state=42
)

print('Columnas originales:')
print(X_train2.dtypes)

In [ ]:
# ── Aplicar One-Hot Encoding ─────────────────────────────────────────────────

columnas_cat = ['tipo_vivienda', 'barrio']
columnas_num = ['metros_cuadrados', 'habitaciones', 'antiguedad', 'distancia_centro']

# OneHotEncoder:
#   drop='first'       → elimina la primera categoría para evitar la Dummy Variable Trap
#   sparse_output=False → devuelve una matriz densa (array normal, no sparse matrix)
#   handle_unknown='ignore' → si aparece una categoría nueva en test, la ignora
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# IMPORTANTE: fit() solo con Train, luego transform() en todos los conjuntos
# Nunca hacer fit con Validation o Test (evita data leakage)
encoder.fit(X_train2[columnas_cat])

# Obtener nombres de las nuevas columnas
nombres_cat = encoder.get_feature_names_out(columnas_cat)
print('Columnas creadas por OHE:')
print(nombres_cat)

In [ ]:
def preparar_features(X, encoder, columnas_num, columnas_cat):
    """Combina columnas numéricas con las columnas OHE en un solo DataFrame."""
    X_cat_encoded = encoder.transform(X[columnas_cat])
    nombres_cat   = encoder.get_feature_names_out(columnas_cat)
    df_cat        = pd.DataFrame(X_cat_encoded, columns=nombres_cat, index=X.index)
    return pd.concat([X[columnas_num].reset_index(drop=True),
                      df_cat.reset_index(drop=True)], axis=1)

X_train_enc = preparar_features(X_train2, encoder, columnas_num, columnas_cat)
X_val_enc   = preparar_features(X_val2,   encoder, columnas_num, columnas_cat)
X_test_enc  = preparar_features(X_test2,  encoder, columnas_num, columnas_cat)

print('Shape tras OHE:', X_train_enc.shape)
print('\nPrimeras filas del dataset codificado:')
X_train_enc.head()

In [ ]:
# ── Entrenar modelo con todas las variables (numéricas + categóricas) ────────
modelo_cat = LinearRegression()
modelo_cat.fit(X_train_enc, y_train2)

# Ver todos los coeficientes
coef_df = pd.DataFrame({
    'Variable':    X_train_enc.columns,
    'Coeficiente': modelo_cat.coef_
}).sort_values('Coeficiente', ascending=False)

print(f'Intercepto: {modelo_cat.intercept_:.2f}\n')
print(coef_df.to_string(index=False))

In [ ]:
# Visualizar la importancia de cada variable
plt.figure(figsize=(10, 6))
colores = ['green' if c > 0 else 'red' for c in coef_df['Coeficiente']]
plt.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colores)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coeficientes del Modelo de Regresión\n(verde = aumenta el precio, rojo = lo reduce)')
plt.xlabel('Coeficiente (miles USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluar en Validation
y_pred_cat = modelo_cat.predict(X_val_enc)
metricas_cat = evaluar_modelo(y_val2, y_pred_cat, 'Validation (con categóricas)')

# Resumen comparativo de los tres modelos
print('═══ COMPARACIÓN FINAL DE MODELOS ════════════════')
print(f'  R² Simple   (1 var):           {metricas_simple["R2"]:.4f}')
print(f'  R² Múltiple (4 vars num):      {metricas_mult["R2"]:.4f}')
print(f'  R² Completo (num + categ):     {metricas_cat["R2"]:.4f}  ← mejor modelo')

## 7. Evaluación Final en Test

Una vez que hemos seleccionado el mejor modelo (el completo con variables categóricas), lo evaluamos **una única vez** en el conjunto de Test para obtener la estimación honesta del rendimiento real:

In [ ]:
# Evaluación final en Test — solo se hace UNA vez
y_pred_test = modelo_cat.predict(X_test_enc)
metricas_test = evaluar_modelo(y_test2, y_pred_test, 'TEST FINAL')

# Residuos: diferencia entre predicción y valor real
residuos = y_test2.values - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico real vs predicho
axes[0].scatter(y_test2, y_pred_test, alpha=0.5, color='steelblue')
lims = [y_test2.min(), y_test2.max()]
axes[0].plot(lims, lims, 'r--', linewidth=2)
axes[0].set_xlabel('Precio Real')
axes[0].set_ylabel('Precio Predicho')
axes[0].set_title(f'Real vs Predicho (Test)\nR² = {metricas_test["R2"]:.4f}')

# Distribución de residuos (deben ser aprox. normales con media 0)
sns.histplot(residuos, kde=True, ax=axes[1], color='coral')
axes[1].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_title(f'Distribución de Residuos\nMedia={residuos.mean():.2f}  Std={residuos.std():.2f}')
axes[1].set_xlabel('Residuo (Real - Predicho)')

plt.tight_layout()
plt.show()

## Ejercicio Final

1. Añade la variable `habitaciones²` (habitaciones al cuadrado) como nuevo feature y entrena un nuevo modelo. ¿Mejora el R²?
2. ¿Qué ocurre si haces `fit` del encoder también con los datos de Validation? ¿Por qué es un problema? (Pista: *data leakage*)
3. Prueba escalando las variables numéricas con `StandardScaler` antes de entrenar. ¿Cambian los coeficientes? ¿Cambia el R²?

In [ ]:
# Escribe tu solución aquí


Haz doble clic **aquí** para ver pistas.

<!--
# 1. Feature polinómico
df['habitaciones_2'] = df['habitaciones'] ** 2
# Repetir la partición y el entrenamiento con la nueva columna

# 3. StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)   # fit solo en train
X_val_scaled   = scaler.transform(X_val_enc)          # solo transform en val
modelo_scaled  = LinearRegression().fit(X_train_scaled, y_train2)
# R² no cambia, pero los coeficientes ahora son comparables entre sí
-->